In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [3]:
# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})

Users data

In [4]:
# users_data = {}
# users_scores = {}

# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})
# mal_client = MALClient(client_id)
# for user in users:
#     user_data = mal_client.get_user_data(user)
#     users_data[user] = user_data
#     scores = mal_client.get_scores(user_data)
#     users_scores[user] = scores

In [5]:
anime_data_client = AnimeDataClient(
    client_id,
    cache_file=PROJECT_ROOT / "anime_cache.json",
)

In [6]:
anime_data = anime_data_client.get_cache()

Build features

In [7]:
# from anime_features import AnimeFeatureBuilder

# builder = AnimeFeatureBuilder(
#     anime_data,
#     max_tfidf_features=3000,
#     n_svd_components=300
# )

# anime_df = builder.build_features()

# builder.svd_explained_variance

Convert each anime in df to vectors

In [8]:
# recommender = SimilarityRecommender()
# anime_vectors = recommender.create_anime_vectors(anime_df)
# anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [9]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Tune SVD components

In [10]:
from anime_evaluation import HitRateEvaluator, RankingMetricEvaluator
from anime_features import AnimeFeatureBuilder

svd_component_results = []

n_runs = 100
max_features = [500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000]
components = [50, 100, 200, 300, 400]
weights_uncertainty = [8.5]
tuning_top_ks = [5, 10]
for n_feature in max_features:
    for component in components:
        builder = AnimeFeatureBuilder(
            anime_data,
            max_tfidf_features=n_feature,
            n_svd_components=component,
        )

        component_anime_df = builder.build_features()

        recommender = SimilarityRecommender()
        recommender.create_anime_vectors(component_anime_df)
        component_anime_df_scaled = recommender.anime_df_scaled

        hitman = HitRateEvaluator(
            anime_df_scaled=component_anime_df_scaled,
            anime_df=component_anime_df,
            scores=user_scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
        )

        (
            bayesian_results,
            bayesian_summary,
            best_bayesian_weights,
            baseline_results,
            baseline_summary,
        ) = hitman.tune_bayesian_uncertainty(
            weights=weights_uncertainty,
            n_runs=n_runs,
            top_ks=tuning_top_ks,
            random_state=42,
        )


        ranking_evaluator = RankingMetricEvaluator(
            anime_df_scaled=component_anime_df_scaled,
            anime_df=component_anime_df,
            scores=user_scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
        )
        _, ranking_summary = ranking_evaluator.evaluate_bayesian(
            uncertainty_weight=weights_uncertainty[0],
            n_runs=n_runs,
            top_ks=tuning_top_ks,
            random_state=42,
            include_global_mean=False,
        )
        ranking_summary = (
            ranking_summary[
                ranking_summary["model"] == "bayesian_ridge"
            ][[
                "k",
                "avg_ndcg_at_k",
                "std_ndcg_at_k",
                "avg_mrr_at_k",
                "avg_relevant_hits_at_k",
                "avg_strong_hits_at_k",
            ]]
        )

        average_metrics = bayesian_summary.merge(
            baseline_summary,
            on="k",
            how="left",
        ).merge(
            ranking_summary,
            on="k",
            how="left",
        ).rename(columns={"uncertainty_weight": "bayesian_uncertainty_weight"})
        average_metrics["component"] = component
        average_metrics["n_feature"] = n_feature
        average_metrics["svd_explained_variance"] = builder.svd_explained_variance

        svd_component_results.append(average_metrics)

svd_component_summary = (
    pd.concat(svd_component_results, ignore_index=True)
    .sort_values(
        ["k", "avg_precision_at_k", "avg_ndcg_at_k"],
        ascending=[True, False, False],
    )
)

svd_component_summary

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,avg_ndcg_at_k,std_ndcg_at_k,avg_mrr_at_k,avg_relevant_hits_at_k,avg_strong_hits_at_k,component,n_feature,svd_explained_variance
74,8.5,5,0.664,0.212498,0.103750,0.033203,3.32,0.102,0.131794,0.015938,0.020593,0.51,0.699978,0.192775,0.990000,3.78,2.89,200,4000,0.285609
56,8.5,5,0.660,0.205971,0.103125,0.032183,3.30,0.102,0.131794,0.015938,0.020593,0.51,0.702499,0.189711,0.990000,3.74,2.93,300,3000,0.410064
84,8.5,5,0.660,0.221108,0.103125,0.034548,3.30,0.102,0.131794,0.015938,0.020593,0.51,0.692799,0.195186,0.990000,3.74,2.87,200,4500,0.274150
64,8.5,5,0.654,0.206667,0.102188,0.032292,3.27,0.102,0.131794,0.015938,0.020593,0.51,0.697179,0.191179,0.990000,3.76,2.89,200,3500,0.298001
76,8.5,5,0.652,0.210089,0.101875,0.032826,3.26,0.102,0.131794,0.015938,0.020593,0.51,0.700833,0.189993,0.990000,3.78,2.93,300,4000,0.373075
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9,8.5,10,0.374,0.126826,0.116875,0.039633,3.74,0.068,0.063373,0.021250,0.019804,0.68,0.448351,0.153302,0.883750,4.67,3.45,400,500,0.928523
91,8.5,10,0.373,0.141318,0.116562,0.044162,3.73,0.068,0.063373,0.021250,0.019804,0.68,0.470768,0.154765,0.898333,4.78,3.57,50,5000,0.104593
11,8.5,10,0.373,0.136222,0.116562,0.042570,3.73,0.068,0.063373,0.021250,0.019804,0.68,0.470583,0.154803,0.879583,4.93,3.65,50,1000,0.203520
31,8.5,10,0.367,0.135628,0.114687,0.042384,3.67,0.068,0.063373,0.021250,0.019804,0.68,0.464127,0.154731,0.899333,4.76,3.50,50,2000,0.150785


## Results

This tuning run evaluates `max_tfidf_features` and SVD components using repeated holdout splits, Bayesian Ridge, and the tuned uncertainty weight. The latest executed sweep uses `bayesian_uncertainty_weight=8.5`. Precision@K is the primary metric, and NDCG@K is included as the ranking-quality tie-breaker when precision is close.

| k | max TF-IDF features | SVD components | Avg precision@k | Std precision@k | Avg hit rate | Avg hits | Avg NDCG@k | Baseline precision@k | SVD explained variance |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 5 | 4000 | 200 | 0.664 | 0.212 | 0.1038 | 3.32 | 0.7000 | 0.102 | 0.2856 |
| 5 | 3000 | 300 | 0.660 | 0.206 | 0.1031 | 3.30 | 0.7025 | 0.102 | 0.4101 |

For `k=5`, the best visible setting by precision is **4000 TF-IDF features** with **200 SVD components**. The previous **3000 TF-IDF features** with **300 SVD components** setting is still very close, trailing by only `0.004` precision while slightly leading on NDCG (`0.7025` vs `0.7000`).

The later `best_svd_components` output below the sweep is stale: it still shows `bayesian_uncertainty_weight=7.5`, while the current sweep uses `8.5`. Rerun that cell after the sweep to refresh the exact best-per-k summary, especially before making a top-10-specific choice.

Conclusion: use **`max_tfidf_features=4000`** and **`n_svd_components=200`** if optimizing strictly for top-5 precision. Keeping **`max_tfidf_features=3000`** and **`n_svd_components=300`** is also defensible because it is nearly tied on precision, has slightly better NDCG in the visible output, and preserves the previous default.
